# Research figure production

This notebook must be adapted to one TODO. It uses complete applicable data from at least one logical source and writes figures plus `figure_manifest.json`; it does not create a Markdown report.


In [ ]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scienceplots  # noqa: F401; registers SciencePlots styles
import seaborn as sns
from scipy import io as scipy_io
from scipy import stats


In [ ]:
TODO_ID = "TODO-XXX"
RAW_PATHS = [Path("REPLACE_WITH_RAW_RESULT_PATH")]
CONDITION_MERGED_PATHS = [Path("REPLACE_WITH_CONDITION_MERGED_PATH")]
CURATED_PATHS: list[Path] = []
FIGURE_DIR = Path("REPLACE_WITH_FIGURE_DIRECTORY")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

STABLE_KEY = ["condition_id"]  # Replace from the TODO.
PREDEFINED_FILTERS: dict[str, object] = {}


In [ ]:
def read_logical_table(path: Path) -> pd.DataFrame:
    if path.is_dir():
        return pd.read_parquet(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported logical table path: {path}")

def read_raw_artifact(path: Path):
    suffix = path.suffix.lower()
    if path.is_dir() or suffix in {".parquet", ".pq", ".csv"}:
        return read_logical_table(path)
    if suffix == ".npy":
        return np.load(path, allow_pickle=False)
    if suffix == ".npz":
        with np.load(path, allow_pickle=False) as data:
            return {name: data[name] for name in data.files}
    if suffix == ".mat":
        try:
            return scipy_io.loadmat(path, simplify_cells=True)
        except NotImplementedError:
            with h5py.File(path, "r") as handle:
                return {name: handle[name][()] for name in handle.keys()}
    if suffix in {".h5", ".hdf5"}:
        with h5py.File(path, "r") as handle:
            return {name: handle[name][()] for name in handle.keys()}
    raise ValueError(f"Unsupported raw artifact path: {path}")

def load_all_tables(paths: list[Path], source_name: str) -> pd.DataFrame:
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing {source_name} paths: {missing}")
    frames = [read_logical_table(path).assign(_source_path=str(path)) for path in paths]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def source_file_list(paths: list[Path]) -> list[str]:
    files: list[str] = []
    for path in paths:
        if path.is_dir():
            files.extend(str(child) for child in sorted(path.rglob("*")) if child.is_file())
        else:
            files.append(str(path))
    return files

def validate_unique_keys(frame: pd.DataFrame, keys: list[str], name: str) -> None:
    absent = [key for key in keys if key not in frame.columns]
    if absent:
        raise KeyError(f"{name} lacks keys: {absent}")
    if frame.duplicated(keys).any():
        raise ValueError(f"{name} has duplicate stable keys")


In [ ]:
missing_raw = [str(path) for path in RAW_PATHS if not path.exists()]
if missing_raw:
    raise FileNotFoundError(f"Missing raw paths: {missing_raw}")
raw_sources = {str(path): read_raw_artifact(path) for path in RAW_PATHS}
condition_merged = load_all_tables(CONDITION_MERGED_PATHS, "condition-merged")
curated = load_all_tables(CURATED_PATHS, "curated") if CURATED_PATHS else pd.DataFrame()
validate_unique_keys(condition_merged, STABLE_KEY, "condition_merged")
print({
    "raw_source_count": len(raw_sources),
    "condition_rows": len(condition_merged),
    "curated_rows": len(curated),
})


## Figure 1 — replace with the TODO-defined question

Use all applicable records from the selected logical source. Compute the estimate at the TODO-defined independent unit before plotting when a high-level plotting API would aggregate incorrectly.


In [ ]:
plot_data = condition_merged.copy()
# Replace x/y/hue and any TODO-defined grouping. Do not cherry-pick rows.
X_COLUMN = "REPLACE_X"
Y_COLUMN = "REPLACE_Y"

with plt.style.context(["science", "no-latex"]):
    fig, ax = plt.subplots(figsize=(6.0, 4.0), constrained_layout=True)
    sns.scatterplot(data=plot_data, x=X_COLUMN, y=Y_COLUMN, ax=ax)
    ax.set_xlabel(X_COLUMN)
    ax.set_ylabel(Y_COLUMN)
    ax.set_title("Replace with a descriptive figure title")
    output_stem = FIGURE_DIR / "figure-01"
    fig.savefig(output_stem.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(output_stem.with_suffix(".png"), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


In [ ]:
manifest = {
    "todo_id": TODO_ID,
    "notebook": Path("research_figures.ipynb").name,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "figures": [
        {
            "figure_id": "figure-01",
            "question_or_metric": "REPLACE_FROM_TODO",
            "source_classes": ["condition-merged"],
            "source_paths": [str(path) for path in CONDITION_MERGED_PATHS],
            "source_files": source_file_list(CONDITION_MERGED_PATHS),
            "stable_key": STABLE_KEY,
            "columns": [X_COLUMN, Y_COLUMN],
            "predefined_filters": PREDEFINED_FILTERS,
            "rows_used": len(plot_data),
            "outputs": ["figure-01.pdf", "figure-01.png"],
        }
    ],
}
(FIGURE_DIR / "figure_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
manifest
